In [1]:
import os
import re
import json
import pickle
import random
import torch
import string
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from copy import deepcopy
from typing import Union, Tuple, Dict
from collections import defaultdict, Counter
from transformers import AutoTokenizer

In [2]:
def pickle_load(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def pickle_dump(path, data):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def json_load(path):
    with open(path, mode='r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def json_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def jsonl_load(path):
    data = []
    with open(path, mode='r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def jsonl_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        for d in data:
            f.write(f"{json.dumps(d, ensure_ascii=False)}\n")

# Process-Supervision

In [238]:
path = "../data/QA_Datasets/msmarco/msmarco_train_full.json"
qa_list = json_load(path)
print(f"qa_list: {len(qa_list)}")

path = "../cache/msmarco/search_cache_train_full_wo_docs_snowflake-llama-3.3-70b.json"
cache_wo_docs = json_load(path)
print(f"cache_wo_docs: {len(cache_wo_docs)}")

qa_list: 37466
cache_wo_docs: 37466


### filtering answer leaked questions

In [239]:
filtered_questions = set()

for qa in qa_list:
    qid = qa['id']
    question = qa['Question']
    answers = qa['answer']
    outputs = cache_wo_docs[question]
    assert len(outputs) == 1
    
    output = outputs[0]['contents'].split('Step 2. Answer:')[-1]
    output = output.lower().strip()

    flag = False
    for answer in answers:
        answer = answer.lower()
        if 'yes' in answer or 'no' in answer:
            continue
        if answer in output:
            flag = True
    if flag:
        continue

    filtered_questions.add(question)
print(f"filtered_questions: {len(filtered_questions)}")

qa_list = [qa for qa in qa_list if qa['Question'] in filtered_questions]
print(f"qa_list: {len(qa_list)}")

filtered_questions: 31796
qa_list: 31796


In [240]:
path = "../cache/msmarco/search_cache_train_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_v1_split.json"
cache_v1 = json_load(path)
print(f"cache_v1: {len(cache_v1)}")

path = "../cache/msmarco/search_cache_train_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_v2_split.json"
cache_v2 = json_load(path)
print(f"cache_v2: {len(cache_v2)}")

path = "../cache/msmarco/search_cache_train_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_v3_split.json"
cache_v3 = json_load(path)
print(f"cache_v3: {len(cache_v3)}")

cache_v1: 37466
cache_v2: 37466
cache_v3: 37466


In [241]:
path = "../data/QA_Datasets/msmarco/answer_logprobs/"
path += "search_cache_train_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_v1_split_answer_logprobs.json"
logprob_list_v1 = json_load(path)
qid2logprob_v1 = {qa['id']:qa['avg_log_prob'] for qa in logprob_list_v1}
print(f"qid2logprob_v1: {len(qid2logprob_v1)}")

path = "../data/QA_Datasets/msmarco/answer_logprobs/"
path += "search_cache_train_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_v2_split_answer_logprobs.json"
logprob_list_v2 = json_load(path)
qid2logprob_v2 = {qa['id']:qa['avg_log_prob'] for qa in logprob_list_v2}
print(f"qid2logprob_v2: {len(qid2logprob_v2)}")

path = "../data/QA_Datasets/msmarco/answer_logprobs/"
path += "search_cache_train_full_conditional_cot2_rewritten_snowflake-llama-3.3-70b_v3_split_answer_logprobs.json"
logprob_list_v3 = json_load(path)
qid2logprob_v3 = {qa['id']:qa['avg_log_prob'] for qa in logprob_list_v3}
print(f"qid2logprob_v3: {len(qid2logprob_v3)}")

qid2logprob_v1: 37466
qid2logprob_v2: 37466
qid2logprob_v3: 37466


### rejection sampling

In [242]:
rejection_sampling_cache = {}

for qa in qa_list:
    qid = qa['id']
    question = qa['Question']
    
    logprob_v1 = qid2logprob_v1[qid]
    logprob_v2 = qid2logprob_v2[qid]
    logprob_v3 = qid2logprob_v3[qid]

    docs = []
    if max(logprob_v1, logprob_v2, logprob_v3) == logprob_v1:
        docs = cache_v1[question]
    elif max(logprob_v1, logprob_v2, logprob_v3) == logprob_v2:
        docs = cache_v2[question]
    elif max(logprob_v1, logprob_v2, logprob_v3) == logprob_v3:
        docs = cache_v3[question]
    assert len(docs) >= 1, f"{logprob_v1}; {logprob_v2}; {logprob_v3}; {qid}"
    rejection_sampling_cache[question] = docs

print(f"qa_list: {len(qa_list)}")
print(f"rejection_sampling_cache: {len(rejection_sampling_cache)}")

qa_list: 31796
rejection_sampling_cache: 31796


In [16]:
# path = "../cache/msmarco/search_cache_train_full_conditional_cot2_rejection_sampling.json"
# json_dump(path, rejection_sampling_cache)

In [243]:
logprob_list = []
for qa in qa_list:
    qid = qa['id']
    question = qa['Question']
    
    logprob_v1 = qid2logprob_v1[qid]
    logprob_v2 = qid2logprob_v2[qid]
    logprob_v3 = qid2logprob_v3[qid]
    logprob_list.append(max(logprob_v1, logprob_v2, logprob_v3))

logprob_list = sorted(logprob_list)
q3 = np.quantile(logprob_list, q=0.25)
print(f"q3 logprob: {q3}")

q3 logprob: -7.88255


In [244]:
rejection_sampling_cache_q3 = {}

for qa in qa_list:
    qid = qa['id']
    question = qa['Question']
    
    logprob_v1 = qid2logprob_v1[qid]
    logprob_v2 = qid2logprob_v2[qid]
    logprob_v3 = qid2logprob_v3[qid]

    if max(logprob_v1, logprob_v2, logprob_v3) < q3:
        continue

    docs = []
    if max(logprob_v1, logprob_v2, logprob_v3) == logprob_v1:
        docs = cache_v1[question]
    elif max(logprob_v1, logprob_v2, logprob_v3) == logprob_v2:
        docs = cache_v2[question]
    elif max(logprob_v1, logprob_v2, logprob_v3) == logprob_v3:
        docs = cache_v3[question]
    assert len(docs) >= 1, f"{logprob_v1}; {logprob_v2}; {logprob_v3}; {qid}"
    rejection_sampling_cache_q3[question] = docs

print(f"qa_list: {len(qa_list)}")
print(f"rejection_sampling_cache_q3: {len(rejection_sampling_cache_q3)}")

qa_list: 31796
rejection_sampling_cache_q3: 23847


In [245]:
# path = "../cache/msmarco/search_cache_train_full_conditional_cot2_rejection_sampling_q3.json"
# json_dump(path, rejection_sampling_cache_q3)

# Preference Optimization

### SFT dataset

In [269]:
path = "../outputs/msmarco.llama-3.2-3b.rag/"
path += "search_cache_train_full_conditional_cot2_rejection_sampling_per_doc/results.json"
rewrite_results = json_load(path)
print(f"rewrite_results: {len(rewrite_results)}")

rewrite_results: 315968


In [271]:
question2did2f1 = defaultdict(dict)

for item in rewrite_results:
    did = item['docid']
    question = item['Question']
    f1 = item['Metrics']['f1']
    question2did2f1[question][did] = f1
    
print(f"question2did2f1: {len(question2did2f1)}")

question2did2f1: 31796


In [273]:
path = "../cache/msmarco/search_cache_train_full_conditional_cot2_rejection_sampling_q3.json"
cache = json_load(path)
print(f"cache: {len(cache)}")

cache: 23847


In [274]:
cache_win = {}
cache_lose = {}

for question, docs in cache.items():
    win_docs = []
    lose_docs = []
    
    for doc in docs:
        did = doc['id']
        f1 = question2did2f1[question][did]
        if f1 == 1:
            win_docs.append(doc)
        if f1 == 0:
            lose_docs.append(doc)

    if len(win_docs) > 0:
        cache_win[question] = win_docs
    if len(lose_docs) > 0:
        cache_lose[question] = lose_docs

print(f"cache_win: {len(cache_win)}")
print(f"cache_lose: {len(cache_lose)}")

cache_win: 14548
cache_lose: 16508


In [260]:
path = "../data/QA_Datasets/msmarco/msmarco_train_full.json"
qa_list = json_load(path)
print(f"qa_list: {len(qa_list)}")

qa_list: 37466


In [261]:
def get_input_prompt(documents, question):
    user_prompt = (
        'You are a helpful assistant. Your job is to analyze the documents below and rewrite only the parts that help clarify or refine the information in relation to the question.\n'
        'List each relevant document to better support answering the question. Do not include unrelated documents.\n\n'

        'Question:\n'
        f'{question}\n\n'
        
        'Documents:\n'
        f'{documents}\n\n'
    )
    return user_prompt

In [262]:
sft_data = []
top_k = 10

for qa in qa_list:
    qid = qa['id']
    question = qa['Question']

    if question not in cache:
        continue
    if question not in cache_win:
        continue
    
    raw_docs = cache[question]
    
    document = ""
    docid2position = {}
    for i, doc in enumerate(raw_docs[:top_k]):
        docid = doc['id']
        content = doc['contents']
        docid2position[docid] = i
        document += f"Document {i + 1}:\n"
        document += f"{content}\n\n"
    input_prompt = get_input_prompt(document, question)
    
    win_docs = cache_win[question]
    rewritten_document = ""
    rewritten_document += '<rewritten_docs>\n'
    for doc in win_docs[:top_k]:
        docid = doc['id']
        content = doc['contents']
        
        position = docid2position[docid]
        rewritten_document += f"Document {position + 1}:\n"
        rewritten_document += f"{content}\n\n"
        
    rewritten_document = rewritten_document.strip() + '\n'
    rewritten_document += '</rewritten_docs>\n'

    sft_data.append({
        'id': qid,
        'conversations': [
            {'from': 'human', 'id':qid, 'value': input_prompt}, 
            {'from': 'llama-3.3-70B-Instruct', 'value': rewritten_document}, 
        ]
    })
print(f"sft_data: {len(sft_data)}")

sft_data: 14548


In [259]:
path = "/data/FIRST/datasets/msmarco_sft.jsonl"
jsonl_dump(path, sft_data)

### DPO dataset

In [418]:
path = "../outputs/msmarco.llama-3.2-3b.rag/"
path += "search_cache_train_full_conditional_cot2_rejection_sampling_per_doc/results.json"
rewrite_results = json_load(path)
print(f"rewrite_results: {len(rewrite_results)}")

rewrite_results: 315968


In [419]:
question2did2f1 = defaultdict(dict)

for item in rewrite_results:
    did = item['docid']
    question = item['Question']
    f1 = item['Metrics']['f1']
    question2did2f1[question][did] = f1
    
print(f"question2did2f1: {len(question2did2f1)}")

question2did2f1: 31796


In [420]:
path = "../cache/msmarco/search_cache_train_full_conditional_cot2_rejection_sampling_q3.json"
cache = json_load(path)
print(f"cache: {len(cache)}")

cache: 23847


In [421]:
cache_win = {}
cache_lose = {}

for question, docs in cache.items():
    win_docs = []
    lose_docs = []
    
    for doc in docs:
        did = doc['id']
        f1 = question2did2f1[question][did]
        if f1 == 1:
            win_docs.append(doc)
        if f1 == 0:
            lose_docs.append(doc)

    if len(win_docs) > 0:
        cache_win[question] = win_docs
    if len(lose_docs) > 0:
        cache_lose[question] = lose_docs

print(f"cache_win: {len(cache_win)}")
print(f"cache_lose: {len(cache_lose)}")

cache_win: 14548
cache_lose: 16508


In [422]:
path = "../data/QA_Datasets/msmarco/msmarco_train_full.json"
qa_list = json_load(path)
print(f"qa_list: {len(qa_list)}")

qa_list: 37466


In [423]:
def get_input_prompt(documents, question):
    user_prompt = (
        'You are a helpful assistant. Your job is to analyze the documents below and rewrite only the parts that help clarify or refine the information in relation to the question.\n'
        'List each relevant document to better support answering the question. Do not include unrelated documents.\n\n'

        'Question:\n'
        f'{question}\n\n'
        
        'Documents:\n'
        f'{documents}\n\n'
    )
    return user_prompt

def wrap_input_documents(docs, top_k=10):
    documents = ""
    for i, doc in enumerate(docs[:top_k]):
        documents += f"Document {i + 1}:\n"
        documents += f"{doc.get('contents', '')}\n\n"
    return documents

def wrap_output_documents(docs, did2position, top_k=10):
    rewritten_document = ""
    rewritten_document += '<rewritten_docs>\n'
    for doc in docs[:top_k]:
        did = doc['id']
        position = did2position[did]
        rewritten_document += f"Document {position + 1}:\n"
        rewritten_document += f"{doc['contents']}\n\n"
    rewritten_document = rewritten_document.strip() + '\n'
    rewritten_document += '</rewritten_docs>\n'
    return rewritten_document

In [428]:
random.seed(42)
dpo_data = []

for qa in qa_list:
    qid = qa['id']
    question = qa['Question']

    if question not in cache:
        continue
    if question not in cache_win:
        continue
    if question not in cache_lose:
        continue

    raw_docs = cache[question]
    win_docs = cache_win[question]
    lose_docs = cache_lose[question]

    raw_dids = [doc['id'] for doc in raw_docs]
    win_dids = [doc['id'] for doc in win_docs]
    lose_dids = [doc['id'] for doc in lose_docs]
    
    dids = win_dids + lose_dids
    remain_dids = [doc['id'] for doc in raw_docs if doc['id'] not in dids]

    if len(remain_dids) > 0:
        n = len(remain_dids)
        k = random.randint(0, n)
        added_dids = random.sample(remain_dids, k)
        dids += added_dids
    dids = set(dids)
    raw_docs = [doc for doc in raw_docs if doc['id'] in dids]


    did2position = {doc['id']:idx for idx, doc in enumerate(raw_docs)}
    documents = wrap_input_documents(raw_docs)
    
    input_prompt = get_input_prompt(documents, question)
    chosen = wrap_output_documents(win_docs, did2position)
    rejected = wrap_output_documents(lose_docs, did2position)
    
    dpo_data.append({'prompt':input_prompt, 'chosen':chosen, 'rejected':rejected})
    
print(f"dpo_data: {len(dpo_data)}")

dpo_data: 8483


In [425]:
# path = "../dpo-train/datasets/msmarco_dpo.jsonl"
# jsonl_dump(path, dpo_data)